# Rankings e comparações entre Copas

## Task 3: montar os rankings e as comparações entre Copas que o time de analytics pediu

## As perguntas do time de analytics

1. Qual seleção **mais marcou gols** em cada Copa?
2. Ela marcou mais ou menos que **na Copa anterior**?

A célula abaixo já vem pronta: o mesmo código do vídeo 3.1, com as cinco views do projeto.

In [ ]:
# vídeo 3.1: as tabelas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)
por_selecao = partidas_por_selecao(partidas)
resumo = resumo_por_selecao(por_selecao)

# vídeo 2.4: escalações e Copas
from wc_pipeline.schemas import PLAYERS, WORLD_CUPS

escalacoes = (
    spark.read.option("header", "true").schema(PLAYERS).csv(f"{RAW}/WorldCupPlayers.csv")
    .select(
        F.col("MatchID").alias("match_id"),
        F.col("Team Initials").alias("sigla"),
        F.col("Player Name").alias("jogador"),
        F.col("Position").alias("posicao"),
    )
)

copas = (
    spark.read.option("header", "true").schema(WORLD_CUPS).csv(f"{RAW}/WorldCups.csv")
    .select(
        F.col("Year").alias("ano"),
        F.col("Country").alias("sede"),
        F.col("Winner").alias("campeao"),
    )
)

# vídeo 3.1: as views
partidas.createOrReplaceTempView("partidas")
por_selecao.createOrReplaceTempView("por_selecao")
resumo.createOrReplaceTempView("resumo")
escalacoes.createOrReplaceTempView("escalacoes")
copas.createOrReplaceTempView("copas")

Uma parte do resumo, para os rankings:

## O que uma window faz

- **`groupBy`**: junta as linhas de cada grupo **numa só**.
- **window**: calcula olhando o grupo, mas **mantém todas as linhas**, com o resultado numa coluna nova.

| Parte | Pergunta | No nosso caso |
|---|---|---|
| `partitionBy` | quais linhas formam um grupo? | as seleções de uma **mesma Copa** |
| `orderBy` | em que ordem, dentro do grupo? | dos **mais gols** para os menos |

## Ranking com empate: três funções

- **`row_number`**: 1, 2, 3, 4 (desempata de forma arbitrária).
- **`rank`**: 1, 2, 2, 4 (empate pula a posição seguinte).
- **`dense_rank`**: 1, 2, 2, 3 (empate não pula).

## A seleção que mais marcou em cada Copa

## Comparando com a Copa anterior: `lag`

Janela por **seleção**, ordenada por **ano**. `lag` pega o valor da linha anterior; `lead`, o da próxima.

Sem Copa anterior, `lag` devolve `null`. Ele pega a Copa anterior **que a seleção disputou**.

## Um acumulado

Soma do início do grupo até a linha atual: o **frame**, declarado com `rowsBetween`.

## A mesma coisa em SQL, com `QUALIFY`

`OVER (PARTITION BY ... ORDER BY ...)`; o `QUALIFY` filtra pelo resultado da window, sem subconsulta.

## Resultado da task 3

A seleção que mais marcou em cada Copa e a comparação com a anterior, em Python e em SQL. Próxima task: publicar uma
tabela.

## Para consultar depois

- **Frame padrão:** com `orderBy` e sem `rowsBetween`, o Spark usa um frame por faixa de valores, e linhas empatadas
  no `orderBy` recebem o mesmo acumulado. Com `rowsBetween`, cada linha conta separadamente.
- **Janela sem `orderBy`** vê o grupo inteiro: serve para colocar o total do grupo em cada linha e calcular
  percentuais, por exemplo `F.sum("gols_pro").over(Window.partitionBy("ano"))`.
- **`lead`** é o contrário do `lag`: pega o valor da próxima linha do grupo.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()